In [1]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go

In [2]:
pca_scores_df = pd.read_csv("../data/output/PCA_DOC_SCORES.csv")
pca_scores_df.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
0,-0.031419,-0.009989,-0.001495,-0.046460,0.056736,-0.057502,-0.038844,0.035569,-0.078975,-0.022602
1,-0.089728,-0.039951,0.008606,-0.020549,0.038700,0.018893,-0.063756,0.064408,-0.056541,0.002780
2,-0.022168,0.004308,0.030995,0.032674,0.006030,-0.043585,-0.041368,0.054335,0.006134,0.035799
3,-0.042335,-0.003622,0.008147,0.017568,0.013874,-0.054836,-0.043328,0.050163,-0.059169,0.036634
4,-0.078409,-0.013729,-0.009947,-0.010194,0.008932,-0.030476,-0.022802,0.046401,-0.029534,-0.024311


In [3]:
metadata_df = pd.read_csv("../data/output/tfidf_doc_index.csv")

doc_labels = metadata_df.iloc[:, 0].tolist()

In [4]:
doc_plot_df = pca_scores_df.copy()
doc_plot_df["Document"] = doc_labels

# Extract book number
doc_plot_df["BookNum"] = (
    doc_plot_df["Document"]
    .str.split("_")
    .str[0]
    .astype(int)
)

# Load book key
key_df = pd.read_csv("../data/archive/key_english.csv")

book_map = dict(zip(key_df["b"], key_df["n"]))
testament_map = dict(zip(key_df["b"], key_df["t"]))

doc_plot_df["BookName"] = doc_plot_df["BookNum"].map(book_map)
doc_plot_df["Testament"] = doc_plot_df["BookNum"].map(testament_map)

doc_plot_df.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,Document,BookNum,BookName,Testament
0,-0.031419,-0.009989,-0.001495,-0.046460,0.056736,-0.057502,-0.038844,0.035569,-0.078975,-0.022602,1_1_1,1,Genesis,OT
1,-0.089728,-0.039951,0.008606,-0.020549,0.038700,0.018893,-0.063756,0.064408,-0.056541,0.002780,1_1_2,1,Genesis,OT
2,-0.022168,0.004308,0.030995,0.032674,0.006030,-0.043585,-0.041368,0.054335,0.006134,0.035799,1_1_3,1,Genesis,OT
3,-0.042335,-0.003622,0.008147,0.017568,0.013874,-0.054836,-0.043328,0.050163,-0.059169,0.036634,1_1_4,1,Genesis,OT
4,-0.078409,-0.013729,-0.009947,-0.010194,0.008932,-0.030476,-0.022802,0.046401,-0.029534,-0.024311,1_1_5,1,Genesis,OT


In [7]:
import plotly.graph_objects as go
import numpy as np

fig_doc = go.Figure()

# Canonical order
books_sorted = (
    doc_plot_df
    .sort_values("BookNum")
    ["BookName"]
    .unique()
)

for book in books_sorted:

    subset = doc_plot_df[doc_plot_df["BookName"] == book]

    subset_ot = subset[subset["Testament"] == "OT"]
    subset_nt = subset[subset["Testament"] == "NT"]

    # Old Testament (filled)
    if not subset_ot.empty:
        fig_doc.add_trace(
            go.Scatter(
                x=subset_ot["PC3"],
                y=subset_ot["PC4"],
                mode="markers",
                name=book,
                marker=dict(size=5, opacity=0.7),
                text=subset_ot["Document"],
                customdata=np.stack(
                    (subset_ot["BookName"], subset_ot["Testament"]),
                    axis=-1
                ),
                hovertemplate=(
                    "Document: %{text}<br>"
                    "Book: %{customdata[0]}<br>"
                    "Testament: %{customdata[1]}<br>"
                    "PC3: %{x:.3f}<br>"
                    "PC4: %{y:.3f}<extra></extra>"
                )
            )
        )

    # New Testament (hollow)
    if not subset_nt.empty:
        fig_doc.add_trace(
            go.Scatter(
                x=subset_nt["PC3"],
                y=subset_nt["PC4"],
                mode="markers",
                name=book if subset_ot.empty else None,
                marker=dict(size=5, opacity=0.7, symbol="circle-open"),
                text=subset_nt["Document"],
                customdata=np.stack(
                    (subset_nt["BookName"], subset_nt["Testament"]),
                    axis=-1
                ),
                hovertemplate=(
                    "Document: %{text}<br>"
                    "Book: %{customdata[0]}<br>"
                    "Testament: %{customdata[1]}<br>"
                    "PC3: %{x:.3f}<br>"
                    "PC4: %{y:.3f}<extra></extra>"
                ),
                showlegend=subset_ot.empty
            )
        )

fig_doc.update_layout(
    title="Documents in PC3–PC4 Space<br>Filled = Old Testament, Hollow = New Testament",
    width=1400,
    height=900,
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.25,
        xanchor="center",
        x=0.5,
        font=dict(size=9)
    ),
    xaxis_title="PC3",
    yaxis_title="PC4"
)

fig_doc

In [8]:
save_path = "/Users/nicholasthornton/Downloads/DS 5001/DS-5001-Bible-Analysis-Final-Project/graphs"

os.makedirs(save_path, exist_ok=True)

fig_doc.write_html(
    os.path.join(save_path, "PCA_documents_PC3_PC4_by_book_and_testament.html")
)

print("PC3–PC4 document PCA plot saved.")

PC3–PC4 document PCA plot saved.


In [12]:
import plotly.graph_objects as go

# First column contains component names
loadings_df = pd.read_csv("../data/output/PCA_LOADINGS.csv")

# Rename first column
loadings_df = loadings_df.rename(columns={loadings_df.columns[0]: "Component"})

# Set component as index
loadings_df = loadings_df.set_index("Component")

# Transpose so rows = terms, columns = PCs
loadings_t = loadings_df.T
loadings_t.index.name = "Term"
loadings_t = loadings_t.reset_index()

loadings_t.head()

Component,Term,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
0,blemish,-0.000500,0.005111,0.008011,-0.000024,0.003619,-0.000830,0.000066,0.006489,-0.001942,-0.001240
1,thousands,-0.002893,-0.001788,-0.003032,-0.002080,-0.000017,0.000856,0.001726,0.000129,-0.001003,-0.003051
2,howbeit,-0.001430,-0.000375,-0.001659,0.002674,-0.003320,-0.002115,-0.001977,-0.001904,-0.000626,0.002139
3,fail,0.004343,0.002338,0.003884,-0.002155,0.001156,0.003009,-0.000051,-0.000088,-0.001711,-0.000785
4,parts,-0.000852,0.000739,0.001497,-0.002319,-0.000273,0.001180,-0.001529,0.001160,-0.002264,0.001414


In [13]:
fig_load = go.Figure()

fig_load.add_trace(
    go.Scatter(
        x=loadings_t["PC3"],
        y=loadings_t["PC4"],
        mode="markers",
        marker=dict(size=5, opacity=0.6),
        text=loadings_t["Term"],
        hovertemplate=
            "Term: %{text}<br>"
            "PC3: %{x:.3f}<br>"
            "PC4: %{y:.3f}<extra></extra>"
    )
)

# Quadrant lines
fig_load.add_shape(
    type="line",
    x0=0, x1=0,
    y0=loadings_t["PC4"].min(),
    y1=loadings_t["PC4"].max(),
    line=dict(color="black", width=1)
)

fig_load.add_shape(
    type="line",
    y0=0, y1=0,
    x0=loadings_t["PC3"].min(),
    x1=loadings_t["PC3"].max(),
    line=dict(color="black", width=1)
)

fig_load.update_layout(
    title="PCA Loadings: PC3 vs PC4",
    xaxis_title="PC3",
    yaxis_title="PC4",
    width=1000,
    height=800
)

fig_load

In [14]:
save_path = "/Users/nicholasthornton/Downloads/DS 5001/DS-5001-Bible-Analysis-Final-Project/graphs"

fig_load.write_html(
    os.path.join(save_path, "PCA_loadings_PC3_PC4.html")
)

print("PC3–PC4 loadings plot saved.")

PC3–PC4 loadings plot saved.


In [15]:
# Sort by PC3 loading
pc3_sorted = loadings_t.sort_values("PC3")

# Top negative terms
print("Top 10 negative terms for PC3:")
print(pc3_sorted.head(10)[["Term", "PC3"]])

print("\nTop 10 positive terms for PC3:")
print(pc3_sorted.tail(10)[["Term", "PC3"]])

Top 10 negative terms for PC3:
Component  Term       PC3
992           i -0.264248
974          my -0.199707
972          ye -0.190883
958         you -0.173978
970          me -0.167612
965        have -0.160947
987        unto -0.114788
935        your -0.112496
981        them -0.107372
966        will -0.097447

Top 10 positive terms for PC3:
Component   Term       PC3
969         thee  0.112924
993           he  0.115308
988            a  0.151662
980           it  0.152496
991          his  0.186723
931        shalt  0.216831
984           be  0.237456
976          thy  0.264598
979         thou  0.287070
997        shall  0.366505


In [16]:
# Sort documents by PC3 score
doc_pc3_sorted = doc_plot_df.sort_values("PC3")

print("Lowest PC3 documents:")
print(doc_pc3_sorted.head(10)[["Document", "BookName", "Testament", "PC3"]])

print("\nHighest PC3 documents:")
print(doc_pc3_sorted.tail(10)[["Document", "BookName", "Testament", "PC3"]])

Lowest PC3 documents:
       Document  BookName Testament       PC3
25892  42_22_28      Luke        NT -0.322956
13311  18_19_14       Job        OT -0.301105
18045  23_21_10    Isaiah        OT -0.300600
23476  40_11_17   Matthew        NT -0.297131
26696  43_14_28      John        NT -0.287179
9165   11_12_14   1 Kings        OT -0.281803
6428     6_22_2    Joshua        OT -0.279537
16423   20_1_23  Proverbs        OT -0.276945
19132   24_7_13  Jeremiah        OT -0.276097
19985  24_42_10  Jeremiah        OT -0.275881

Highest PC3 documents:
       Document     BookName Testament       PC3
13235  18_15_32          Job        OT  0.359728
18858   23_62_4       Isaiah        OT  0.360356
3267    3_18_16    Leviticus        OT  0.370472
2330    2_28_37       Exodus        OT  0.373428
12975   18_5_24          Job        OT  0.374407
13416  18_22_27          Job        OT  0.380041
8807    11_2_37      1 Kings        OT  0.382571
16502   20_4_12     Proverbs        OT  0.424560
1617   

In [17]:
book_pc3_means = (
    doc_plot_df
    .groupby("BookName")["PC3"]
    .mean()
    .sort_values()
)

print(book_pc3_means)

BookName
1 Thessalonians   -0.066284
Philippians       -0.065335
Haggai            -0.057616
2 Corinthians     -0.055377
Jude              -0.053378
                     ...   
Deuteronomy        0.058772
Proverbs           0.061102
Obadiah            0.069185
Leviticus          0.073329
Nahum              0.082592
Name: PC3, Length: 66, dtype: float64


In [18]:
testament_means = (
    doc_plot_df
    .groupby("Testament")["PC3"]
    .mean()
)

print(testament_means)

Testament
NT   -0.017966
OT    0.006177
Name: PC3, dtype: float64


The third principal component captures a stylistic contrast driven largely by archaic second-person forms versus more general first- and second-person pronouns. Words with strong positive loadings on PC3 include “thou,” “thy,” “shall,” “shalt,” and “thee,” which are characteristic of elevated or archaic address. In contrast, strong negative loadings include more common pronouns such as “I,” “my,” “you,” “me,” and “your,” along with frequently used verbs like “have” and “will.” This suggests that PC3 distinguishes passages that rely heavily on archaic second-person constructions from those using more standard personal pronouns. At the document level, the highest PC3 scores are concentrated in Old Testament books such as Deuteronomy, Exodus, Proverbs, and Job, while lower values appear more frequently in New Testament writings. Overall, the polarity reflects a difference in stylistic register and pronoun usage rather than a broad theological or testament-level divide, as the average PC3 values for OT and NT are relatively close.